<a href="https://colab.research.google.com/github/sumair789-lgtm/urdu-ocr-codesaviours-si26--Sumair-/blob/main/SI26-Week4-Sumair.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Week 4 Tasks

Install Library


In [2]:
!pip install transformers torch pillow pandas sentencepiece -q

Mount Drive

In [2]:
import os
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/urdu-ocr-si26'
csv_path = os.path.join(base_path, 'data', 'labels.csv')
print('Base path exists:', os.path.exists(base_path))
print('CSV path exists:', os.path.exists(csv_path))

Mounted at /content/drive
Base path exists: True
CSV path exists: True


Dataset Class

In [3]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import pandas as pd

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, processor, max_target_length=128):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        self.max_target_length = max_target_length
        print(f"Dataset loaded: {len(self.data)} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = Image.open(row['image']).convert('RGB')
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()

        labels = self.processor.tokenizer(
            row['text'],
            padding="max_length",
            max_length=self.max_target_length,
            truncation=True
        ).input_ids
        labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]

        return {"pixel_values": pixel_values, "labels": torch.tensor(labels)}

Build processor + dataset (verified path baked in)

In [4]:
from transformers import TrOCRProcessor, ViTImageProcessor, RobertaTokenizer
import os

# Same method as your Week 3 notebook — avoids the sentencepiece/fast-tokenizer error
image_processor = ViTImageProcessor.from_pretrained('microsoft/trocr-base-printed')
tokenizer = RobertaTokenizer.from_pretrained('microsoft/trocr-base-printed')
processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)

csv_path = '/content/drive/MyDrive/urdu-ocr-si26/data/labels.csv'
dataset = UrduOCRDataset(csv_path, processor)

if 'Unnamed: 1' in dataset.data.columns:
    dataset.data = dataset.data.rename(columns={'Unnamed: 1': 'text'})

drive_base_path = '/content/drive/MyDrive/urdu-ocr-si26'
dataset.data['image'] = dataset.data['image'].apply(
    lambda x: os.path.join(drive_base_path, x) if not str(x).startswith('/content') else x
)

sample = dataset[0]
print('Sample pixel_values shape:', sample['pixel_values'].shape)
print('Sample labels shape:', sample['labels'].shape)
print('Dataset is working correctly!')

preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Dataset loaded: 273 samples
Sample pixel_values shape: torch.Size([3, 384, 384])
Sample labels shape: torch.Size([128])
Dataset is working correctly!


Train/Test Split

In [6]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

print(f'Total samples: {len(dataset)}')
print(f'Training samples: {train_size}')
print(f'Testing samples: {test_size}')

Total samples: 273
Training samples: 218
Testing samples: 55


Load Model

In [7]:
from transformers import VisionEncoderDecoderModel
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cpu':
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU')

model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed')
model = model.to(device)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print('Model loaded successfully!')
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

Using device: cpu


config.json:   0%|          | 0.00/4.13k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.33GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-printed
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!
Model parameters: 333,921,792


Training Setup

In [8]:
from torch.utils.data import DataLoader
from torch.optim import AdamW

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4)
optimizer = AdamW(model.parameters(), lr=5e-5)

print(f'Training batches per epoch: {len(train_loader)}')
print('Ready to train!')

Training batches per epoch: 55
Ready to train!


Loop

In [14]:
num_epochs = 40
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    print(f'\nEpoch {epoch + 1}/{num_epochs}')
    print('-' * 30)

    for batch_idx, batch in enumerate(train_loader):
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        if batch_idx % 10 == 0:
            print(f'  Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}')

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch + 1} complete | Average Loss: {avg_loss:.4f}')

print('\nTraining complete!')


Epoch 1/40
------------------------------
  Batch 0/55 | Loss: 3.0953
  Batch 10/55 | Loss: 2.9003
  Batch 20/55 | Loss: 3.1282
  Batch 30/55 | Loss: 3.1032
  Batch 40/55 | Loss: 3.3749
  Batch 50/55 | Loss: 3.0789
Epoch 1 complete | Average Loss: 3.0980

Epoch 2/40
------------------------------
  Batch 0/55 | Loss: 3.0508
  Batch 10/55 | Loss: 3.1411
  Batch 20/55 | Loss: 3.0445
  Batch 30/55 | Loss: 2.9126
  Batch 40/55 | Loss: 3.0196
  Batch 50/55 | Loss: 3.0232
Epoch 2 complete | Average Loss: 3.0878

Epoch 3/40
------------------------------
  Batch 0/55 | Loss: 3.0762
  Batch 10/55 | Loss: 3.0692
  Batch 20/55 | Loss: 3.4874
  Batch 30/55 | Loss: 3.2937
  Batch 40/55 | Loss: 3.2232
  Batch 50/55 | Loss: 3.0579
Epoch 3 complete | Average Loss: 3.1445

Epoch 4/40
------------------------------
  Batch 0/55 | Loss: 3.2647
  Batch 10/55 | Loss: 3.0448
  Batch 20/55 | Loss: 3.1648
  Batch 30/55 | Loss: 3.0640
  Batch 40/55 | Loss: 3.4074
  Batch 50/55 | Loss: 3.1694
Epoch 4 complete

Evaluation

In [15]:
model.eval()
print('=== Model Evaluation on Test Images ===\n')

correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].clone()
        labels[labels == -100] = processor.tokenizer.pad_token_id

        generated_ids = model.generate(pixel_values)
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)
        actual_text = processor.batch_decode(labels, skip_special_tokens=True)

        for pred, actual in zip(generated_text, actual_text):
            total += 1
            if pred.strip() == actual.strip():
                correct += 1
            print(f'Predicted: {pred}')
            print(f'Actual: {actual}')
            print()

accuracy = (correct / total) * 100 if total > 0 else 0
print(f'Accuracy: {accuracy:.1f}% ({correct}/{total} correct)')

=== Model Evaluation on Test Images ===

Predicted: ن�����������������
Actual: پاکستانی اور انڈین شخصیات کے درمیان ٹریک ٹو ملاقاتوں اور وزرائے اعظم کو لکھے گئے خط کے حوالے سے

Predicted: ��������������������
Actual: میں سے کسی نے پولیس پر فائرنگ کی۔ انہوں

Predicted: �ی���ااااا�������
Actual: اتنے میں وہ تینوں ہال میں داخل ہوئے ۔ مہمانوں نے تالیاں بجائیں جن کا جواب تینوں نے یوں ہاتھ ہلا ہلا �

Predicted: ا�������������������
Actual: اپنے مظاہروں کا سلسلہ ختم کر دیں۔ انہوں نے تمام تنظیمی ونگز

Predicted: ووو�    ����������
Actual: محب ایک دفعہ ایک عورت کا بچہ گم ہو گیا۔ وہ اسے قافلے میں ڈھونڈتی پھر رہی تھی۔ وہ ایک ایک

Predicted: ������������������
Actual: بادشاہوں نے اپنے اقتدار کو جائز قرار دینے اور اسے دوام بخشنے

Predicted: �ااااااااااا������
Actual: مظفر الیکٹرک اینڈ الیکٹرانکس سینٹر

Predicted: �ی���اااا��������
Actual: تعلیم ہر انسان کا حق ہے

Predicted: ������������������
Actual: حج کی ادائیگی کا طری آٹھ ذوالحجہ سے بارہ ذوالحجہ تک حج کے مخصوص پانچ دن ہیں۔ ان پانچ دنوں میں ح

Pre

In [13]:
save_path = '/content/drive/MyDrive/SI26-urdu-ocr-model'
model.save_pretrained(save_path)
processor.save_pretrained(save_path)

print(f'Model saved to Google Drive: {save_path}')
print('You can load this model again next week without retraining')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to Google Drive: /content/drive/MyDrive/SI26-urdu-ocr-model
You can load this model again next week without retraining


In [9]:
print("=== Raw label check (first 3 rows) ===")
for i in range(3):
    print(f"Label {i}: {dataset.data.iloc[i]['text']}")

=== Raw label check (first 3 rows) ===
Label 0: بیرون ملک مقیم پاکستانیوں کی بہبود کیلئے جاری منصوبوں میں مزید تیزی لائی جائے، وزیر اعظم
Label 1: نجکاری کمیشن نے فیسکو اور گیپکو کے لیے اظہار دلچسپی جمع کرانے کی تاریخ میں توسیع کر دی
Label 2: رولیکس نے 1940 کی دہائی میں 'رولیکس 4113' ماڈل کی صرف 12 گھڑیاں ہی بنائی تھیں جن میں سےاب صرف 9 گھڑیاں ہی دنیا میں


In [10]:
!pip install jiwer -q
from jiwer import cer

model.eval()
predicted_texts, actual_texts = [], []

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].clone()
        labels[labels == -100] = processor.tokenizer.pad_token_id

        generated_ids = model.generate(pixel_values)
        predicted_texts.extend(processor.batch_decode(generated_ids, skip_special_tokens=True))
        actual_texts.extend(processor.batch_decode(labels, skip_special_tokens=True))

score = cer(actual_texts, predicted_texts)
print(f'Character Error Rate (CER): {score:.4f} ({score*100:.1f}%)')
print("\n=== Sample comparison ===")
for i in range(5):
    print(f'Predicted: {predicted_texts[i]}')
    print(f'Actual: {actual_texts[i]}')
    print()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 54.3 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1625: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Character Error Rate (CER): 1.0012 (100.1%)

=== Sample comparison ===
Predicted: SALES:
Actual: مسلمان بھی اللہ تعالیٰ محبوب اور نبی آخرالزمان حضرت

Predicted: CSTALL:$168.00
Actual: مال اور طولِ عمر کی حرص حضرت انس رضی اللہ عنہ نے فرمایا نبی کریم صلی اللہ علیہ وسلم

Predicted: =
Actual: رولیکس نے 1940 کی دہائی میں 'رولیکس 4113' ماڈل کی صرف 12 گھڑیاں ہی بنائی تھیں جن میں سےاب صرف 9 گھڑیاں ہ

Predicted: EXCLUDING AND EXCLUDES ON BACK
Actual: ایف آئی آر میں کیا لکھا ہے؟

Predicted: CARD SALAD
Actual: دَرَخْت - بُلْبُل - اُداس - جُگْنُو - رَوشْنِی

